## Task A2: Lagged Exposure Analysis Across Mutations (2 points)


### Mutation Selection

We analyse three cell lines:

- **WT** — wild-type MCF10A; baseline ERK dynamics with no PI3K–AKT pathway alteration.
- **AKT1_E17K** — gain-of-function point mutation directly in AKT1; constitutively active AKT independently of PI3K.
- **PTEN_del** — deletion of the PTEN phosphatase, which normally degrades PIP3; its loss leads to chronically elevated PIP3 and secondary AKT hyperactivation.

Both mutants hyperactivate AKT through mechanistically distinct nodes (direct effector vs. loss of a negative regulator), making them informative for comparing whether ERK relay kinetics depend on the mechanism of AKT activation.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# locate the scripts directory relative to this notebook
_nb_dir = Path('.').resolve()
_candidates = [
    _nb_dir / 'spatiotemporal-continuation' / 'spatiotemporal-continuation' / 'scripts',
    _nb_dir / 'scripts',
    _nb_dir.parent / 'scripts',
]
for _c in _candidates:
    if _c.exists():
        sys.path.insert(0, str(_c))
        print(f'Scripts: {_c}')
        break
else:
    print('WARNING: scripts directory not found — check path')

from spatiotemporal_signal_propagation import (
    load_metadata,
    load_site_block,
    add_track_deltas,
    assign_jump_events,
    build_spatial_edges,
    annotate_spatial_exposure,
    resolve_path,
)
print('Imports OK')

Scripts: C:\Users\Yoga\Desktop\biol_sys_projekt2\biolsys_project2\spatiotemporal-continuation\spatiotemporal-continuation\scripts
Imports OK


In [2]:
# ── Analysis parameters ────────────────────────────────────────────────────────
DATA_PATH      = resolve_path(Path('single-cell-tracks_exp1-6_noErbB2.csv.gz'))
META_PATH      = resolve_path(Path('01-readme-experiment-description_2022-04-05.csv'))

SIGNAL_COL     = 'ERKKTR_ratio'
SPATIAL_RADIUS = 60.0
JUMP_QUANTILE  = 0.9
CHUNKSIZE      = 1_000_000

LAGS           = list(range(7))          # τ = 0..6 frames

MUTATIONS      = ['WT', 'AKT1_E17K', 'PTEN_del']
COLORS         = {'WT': '#1f77b4', 'AKT1_E17K': '#ff7f0e', 'PTEN_del': '#2ca02c'}

print('Parameters set.')
print(f'  Signal:         {SIGNAL_COL}')
print(f'  Spatial radius: {SPATIAL_RADIUS} px')
print(f'  Lags τ:         {LAGS} (frames)')

Parameters set.
  Signal:         ERKKTR_ratio
  Spatial radius: 60.0 px
  Lags τ:         [0, 1, 2, 3, 4, 5, 6] (frames)


In [5]:
# ── Load metadata and discover available blocks ────────────────────────────────
meta = load_metadata(META_PATH)
FRAME_MINUTES = float(meta['Acquisition_frequency_min'].iloc[0])
print(f'Acquisition frequency: {FRAME_MINUTES} min/frame')
print(f'Lags in minutes: {[lag * FRAME_MINUTES for lag in LAGS]}\n')

# scan CSV in chunks to find all (Exp_ID, Site) pairs present in the data
_pairs: set = set()
for _chunk in pd.read_csv(DATA_PATH, usecols=['Exp_ID', 'Image_Metadata_Site'], chunksize=CHUNKSIZE):
    _pairs.update(map(tuple, _chunk.drop_duplicates().values.tolist()))

blocks = pd.DataFrame(sorted(_pairs), columns=['Exp_ID', 'Image_Metadata_Site'])
blocks = blocks.merge(meta[['Image_Metadata_Site', 'Mutation']], on='Image_Metadata_Site', how='left')
blocks = blocks[blocks['Mutation'].isin(MUTATIONS)].reset_index(drop=True)

print('Available blocks per mutation:')
print(blocks.groupby('Mutation').size().rename('n_blocks').to_string())

Acquisition frequency: 5.0 min/frame
Lags in minutes: [0.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0]

Available blocks per mutation:
Mutation
AKT1_E17K    21
PTEN_del     27
WT           24


In [4]:
# ── Helper functions ───────────────────────────────────────────────────────────

def compute_lagged_jump_flag(block: pd.DataFrame, lag: int) -> np.ndarray:
    """For each node (cell c, frame t), return whether cell c jumps at frame t+lag."""
    if lag == 0:
        return block['jump_event'].to_numpy(dtype=bool)

    # Shift future records back by lag so they align with the current frame t.
    # Merging on (track_id, Image_Metadata_T) then gives the jump at t+lag for each t.
    future = block[['track_id', 'Image_Metadata_T', 'jump_event']].copy()
    future['Image_Metadata_T'] = future['Image_Metadata_T'] - lag
    future = future.rename(columns={'jump_event': 'future_jump'})

    merged = block[['track_id', 'Image_Metadata_T']].merge(
        future, on=['track_id', 'Image_Metadata_T'], how='left'
    )
    return merged['future_jump'].fillna(False).to_numpy(dtype=bool)


def compute_rr_at_lag(block: pd.DataFrame, lag: int) -> dict:
    """Return poolable counts for RR(τ) for one block at one lag.

    Pooling raw counts across blocks before dividing avoids the bias
    that arises from averaging ratios across blocks of different sizes.
    """
    exposure = block['neighbor_jump_now'].fillna(False).to_numpy(dtype=bool)
    response = compute_lagged_jump_flag(block, lag)
    return {
        'n_exposed':       int(exposure.sum()),
        'n_unexposed':     int((~exposure).sum()),
        'exposed_jumps':   int(response[exposure].sum()),
        'unexposed_jumps': int(response[~exposure].sum()),
    }


def analyze_block(exp_id: int, site_id: int, mutation: str) -> list:
    """Run the full pipeline for one (exp, site) block and return per-lag count dicts."""
    block = load_site_block(DATA_PATH, exp_id, site_id, CHUNKSIZE)
    block, threshold = add_track_deltas(block, SIGNAL_COL, FRAME_MINUTES, JUMP_QUANTILE)
    block = assign_jump_events(block, threshold)
    spatial_edges = build_spatial_edges(block, SPATIAL_RADIUS)
    block = annotate_spatial_exposure(block, spatial_edges)

    rows = []
    for lag in LAGS:
        row = compute_rr_at_lag(block, lag)
        row.update({'exp_id': exp_id, 'site_id': site_id, 'mutation': mutation, 'lag': lag})
        rows.append(row)
    return rows


print('Functions defined.')

Functions defined.


In [ ]:
# ── Main computation loop ──────────────────────────────────────────────────────
# Runs the full pipeline for every (exp, site) block and collects counts.
# Expect ~30 s per block; ~72 blocks total ≈ 35 min.

all_rows = []

for _, info in blocks.iterrows():
    exp_id   = int(info['Exp_ID'])
    site_id  = int(info['Image_Metadata_Site'])
    mutation = info['Mutation']
    print(f'Exp {exp_id}  Site {site_id:2d}  ({mutation}) ...', end='  ')
    try:
        rows = analyze_block(exp_id, site_id, mutation)
        all_rows.extend(rows)
        print('OK')
    except Exception as exc:
        print(f'SKIPPED: {exc}')

results = pd.DataFrame(all_rows)
print(f'\nDone. Total records: {len(results)}')
print(results.groupby('mutation')['lag'].count().rename('n_records').to_string())

In [ ]:
# ── Pooled RR(τ) curves and optimal lag τ* ────────────────────────────────────

def pooled_rr_curve(df: pd.DataFrame) -> pd.DataFrame:
    """Pool counts across all blocks for one mutation, then compute RR per lag.

    Summing counts before dividing means larger blocks contribute proportionally more,
    avoiding Jensen's inequality bias from averaging ratios.
    """
    agg = (
        df.groupby('lag')[['n_exposed', 'n_unexposed', 'exposed_jumps', 'unexposed_jumps']]
        .sum()
        .reset_index()
    )
    agg['exposed_rate']   = agg['exposed_jumps']   / agg['n_exposed'].replace(0, np.nan)
    agg['unexposed_rate'] = agg['unexposed_jumps'] / agg['n_unexposed'].replace(0, np.nan)
    agg['RR']             = agg['exposed_rate']    / agg['unexposed_rate'].replace(0, np.nan)
    agg['lag_min']        = agg['lag'] * FRAME_MINUTES
    return agg


rr_curves = {mut: pooled_rr_curve(results[results['mutation'] == mut]) for mut in MUTATIONS}

tau_star = {}
for mut, curve in rr_curves.items():
    best_idx = curve['RR'].idxmax()
    best     = curve.loc[best_idx]
    tau_star[mut] = {
        'tau_frames': int(best['lag']),
        'tau_min':    float(best['lag_min']),
        'max_RR':     float(best['RR']),
    }

print('Optimal lag τ* (lag where RR is highest):')
print(f'{"Mutation":15s}  {"τ* (frames)":>12s}  {"τ* (min)":>10s}  {"max RR":>8s}')
print('-' * 55)
for mut, v in tau_star.items():
    print(f'{mut:15s}  {v["tau_frames"]:>12d}  {v["tau_min"]:>10.0f}  {v["max_RR"]:>8.3f}')

In [ ]:
# ── Plot: RR(τ) vs τ with markers at τ* ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

x_ticks = [lag * FRAME_MINUTES for lag in LAGS]

for mut in MUTATIONS:
    curve  = rr_curves[mut].set_index('lag')
    y_vals = [float(curve.loc[lag, 'RR']) if lag in curve.index else np.nan for lag in LAGS]

    ax.plot(x_ticks, y_vals, marker='o', ms=5, lw=1.8, color=COLORS[mut], label=mut)

    ts = tau_star[mut]
    ax.plot(
        ts['tau_min'], ts['max_RR'],
        marker='*', ms=16, color=COLORS[mut], zorder=5, linestyle='none',
        label=f'τ* = {ts["tau_min"]:.0f} min  ({mut})',
    )

ax.axhline(1.0, color='gray', ls='--', lw=1, label='RR = 1 (no effect)')
ax.set_xlabel('Lag τ (min)', fontsize=12)
ax.set_ylabel('Relative Risk  RR(τ)', fontsize=12)
ax.set_title(
    'Lagged Exposure RR(τ): ERK signal propagation timescales\n'
    'WT  vs  AKT1_E17K  vs  PTEN_del',
    fontsize=12,
)
ax.set_xticks(x_ticks)
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('A2_lagged_RR.png', dpi=150)
plt.show()
print('Figure saved as A2_lagged_RR.png')

In [7]:
# ── Summary table ──────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    {
        'mutation':                    mut,
        'optimal_lag_tau_star_frames': v['tau_frames'],
        'optimal_lag_tau_star_min':    v['tau_min'],
        'max_RR':                      round(v['max_RR'], 3),
        'n_blocks':                    int(len(results[results['mutation'] == mut]) // len(LAGS)),
    }
    for mut, v in tau_star.items()
])

print('Summary table:')
display(summary)
summary.to_csv('A2_summary_table.csv', index=False)
print('Table saved as A2_summary_table.csv')

NameError: name 'tau_star' is not defined